In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# تحميل الداتا الخاصة بفشل القلب (تأكد من اسم الملف لديك، وغالباً يكون heart_failure_clinical_records_dataset.csv)
df = pd.read_csv('heart_failure_clinical_records_dataset.csv')

# 1. فحص أبعاد الداتا
print("--- Dataset Shape ---")
print(df.shape)

# 2. فحص القيم المفقودة
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# 3. فحص توزيع الهدف (DeathEvent / Target)
print("\n--- Target Distribution ---")
print(df['DEATH_EVENT'].value_counts())

# استعراض أول 5 صفوف
display(df.head())

# 4. معلومات الأعمدة وأنواعها
df.info()

--- Dataset Shape ---
(299, 13)

--- Missing Values Count ---
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

--- Target Distribution ---
DEATH_EVENT
0    203
1     96
Name: count, dtype: int64


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB


In [2]:
# 1. فحص الـ Outliers للأعمدة العددية الأساسية باستخدام IQR
num_cols = ['age', 'creatinine_phosphokinase', 'platelets', 'serum_creatinine', 'ejection_fraction', 'serum_sodium']

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
    print(f"Outliers in {col}: {outliers_count}")

# 2. فصل الـ Features (X) عن الـ Target (y)
X = df.drop('DEATH_EVENT', axis=1)
y = df['DEATH_EVENT']

# 3. تقسيم البيانات (Train-Test Split) مع الـ Stratification
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. توحيد مقياس البيانات (Feature Scaling) باستخدام StandardScaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Heart Failure Data Pipeline Prepared Successfully! ---")
print("X_train scaled shape:", X_train_scaled.shape)
print("X_test scaled shape:", X_test_scaled.shape)

Outliers in age: 0
Outliers in creatinine_phosphokinase: 29
Outliers in platelets: 21
Outliers in serum_creatinine: 29
Outliers in ejection_fraction: 2
Outliers in serum_sodium: 4

--- Heart Failure Data Pipeline Prepared Successfully! ---
X_train scaled shape: (239, 12)
X_test scaled shape: (60, 12)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# 1. تعريف النماذج
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss')
}

print("--- Heart Failure Models Evaluation ---\n")

# 2. حلقة التدريب والتقييم
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])

    print(f"Model: {name}")
    print(f"   Accuracy:  {acc * 100:.2f}%")
    print(f"   ROC-AUC:   {auc:.4f}")
    print("-" * 40)

# 3. تقرير الأداء المفصل لأفضل نموذج
best_model = RandomForestClassifier(random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)

print("\n--- Detailed Classification Report (Random Forest) ---")
print(classification_report(y_test, y_pred_best))

--- Heart Failure Models Evaluation ---

Model: Logistic Regression
   Accuracy:  81.67%
   ROC-AUC:   0.8588
----------------------------------------
Model: Random Forest
   Accuracy:  83.33%
   ROC-AUC:   0.8909
----------------------------------------
Model: XGBoost
   Accuracy:  81.67%
   ROC-AUC:   0.8318
----------------------------------------

--- Detailed Classification Report (Random Forest) ---
              precision    recall  f1-score   support

           0       0.84      0.93      0.88        41
           1       0.80      0.63      0.71        19

    accuracy                           0.83        60
   macro avg       0.82      0.78      0.79        60
weighted avg       0.83      0.83      0.83        60



In [4]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# تطبيق Cross-Validation بـ 5 أجزاء (5-Fold) على نموذج Random Forest
rf_model = RandomForestClassifier(random_state=42)

# حساب الدقة عبر الـ Cross-Validation
cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("--- Cross-Validation Results ---")
print(f"Scores across 5 folds: {cv_scores}")
print(f"Mean Accuracy: {cv_scores.mean() * 100:.2f}%")
print(f"Standard Deviation: {cv_scores.std():.4f}")

--- Cross-Validation Results ---
Scores across 5 folds: [0.8125     0.83333333 0.91666667 0.83333333 0.85106383]
Mean Accuracy: 84.94%
Standard Deviation: 0.0358
